#### Preparing Data

In [1]:
# import the necessary packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import keras

In [2]:
# reading csv data
columns = [
    "capacity",
    "diameter",
    "column_height",
    "porosity",
    "ads_time",
    "C_o",
    "rho_s",
    "dp",
    "qm1",
    "kl1",
    "sorbent_cost_per_kg",
    "total_LCP",
    "emissions",
    "PFAS_eff",
    "PFAS_ads",
    ]

df1 = pd.read_csv("./training_data/ALLMAT_200k_training_data.csv", usecols=columns)
df = df1.dropna(ignore_index=True)
# df.tail()

In [3]:
# count rows before cleaning
print(f"Number of rows before cleaning: {len(df)}")

Number of rows before cleaning: 200000


In [4]:
epsilon = 1e-20
# cleaning data
df.loc[:, 'PFAS_eff'] = df['PFAS_eff'].clip(lower=0)
df = df[(df['total_LCP'] >= 0)].copy()

# Applying log transformations
df['capacity'] = np.log(df['capacity'])
df['ads_time'] = np.log(df['ads_time'])
df['C_o'] = np.log(df['C_o'])
df['dp'] = np.log(df['dp'])
df['qm1'] = np.log(df['qm1'])
df['kl1'] = np.log(df['kl1'])
df['sorbent_cost_per_kg'] = np.log(df['sorbent_cost_per_kg'])
df['total_LCP'] = np.log(df['total_LCP'])
df['emissions'] = np.log(df['emissions'])
df['PFAS_eff'] = np.log(df['PFAS_eff'] + epsilon)

# bounds
print(df.min())
print(df.max())

capacity                 2.304194
diameter                 0.955110
column_height            1.000010
porosity                 0.350001
ads_time                 3.401257
C_o                     -6.815012
rho_s                  500.004000
dp                      -8.111725
qm1                     -6.775754
kl1                      0.003334
sorbent_cost_per_kg      0.003125
total_LCP                9.997128
emissions                5.171745
PFAS_eff               -46.051702
PFAS_ads                 0.000022
dtype: float64
capacity                  8.517189
diameter                  4.000000
column_height             3.999990
porosity                  0.499998
ads_time                  7.509330
C_o                       2.302583
rho_s                  1999.980000
dp                       -5.991469
qm1                       2.302564
kl1                       9.210334
sorbent_cost_per_kg       7.600902
total_LCP                29.519682
emissions                19.495519
PFAS_eff            

In [5]:
# count rows after cleaning
print(f"Number of rows after cleaning: {len(df)}")

Number of rows after cleaning: 200000


In [6]:
# separate the data into inputs and outputs
inputs = [
    "capacity",
    "diameter",
    "column_height",
    "porosity",
    "ads_time",
    "C_o",
    "rho_s",
    "dp",
    "qm1",
    "kl1",
    "sorbent_cost_per_kg",
    ]

outputs = [
    "total_LCP",
    "emissions",
    "PFAS_eff"
]

dfin = df[inputs]
dfout = df[outputs]

In [7]:
# define a function to scale the data using min-max scaling
def min_max_scaler(data, data_min, data_max):
    return (data - data_min) /(data_max - data_min)

# define a function to reverse the scaling
def reverse_min_max_scaler(scaled_data, data_min, data_max):
    return scaled_data * (data_max - data_min) + data_min

In [8]:
# scale the data
x_data_min = dfin.min()
x_data_max = dfin.max()
y_data_min = dfout.min()
y_data_max = dfout.max()

x_data_scaled = min_max_scaler(dfin, x_data_min, x_data_max)
y_data_scaled = min_max_scaler(dfout, y_data_min, y_data_max)

x_scaled_min = x_data_scaled.min()
x_scaled_max = x_data_scaled.max()
y_scaled_min = y_data_scaled.min()
y_scaled_max = y_data_scaled.max()


#### Optimise Model with pyomo

In [9]:
import pyomo.environ as pyo
# omlt for interfacing our neural network with pyomo
from Custom_scaling import MinMaxScaling
from omlt import OffsetScaling, OmltBlock
from omlt.io.keras import keras_reader, load_keras_sequential
from omlt.neuralnet import (
    FullSpaceNNFormulation,
    FullSpaceSmoothNNFormulation,
    ReducedSpaceSmoothNNFormulation,
    ReluBigMFormulation,
    ReluComplementarityFormulation,
    ReluPartitionFormulation,
)
from omlt.neuralnet.activations import ComplementarityReLUActivation

In [10]:
# # Check solver availability
# for solver in ["glpk", "cbc", "ipopt", "baron", "cplex", "appsi_maingo", "gurobi"]:
#     print(solver, pyo.SolverFactory(solver).available())

In [11]:
# unscaling back using factors
scaler = MinMaxScaling(
        min_inputs={i: x_data_min[inputs[i]] for i in range(len(inputs))},
        max_inputs={i: x_data_max[inputs[i]] for i in range(len(inputs))},
        min_outputs={i: y_data_min[outputs[i]] for i in range(len(outputs))},
        max_outputs={i: y_data_max[outputs[i]] for i in range(len(outputs))},
    )

# saving unscaled bounds
#input
unscaled_lb = x_data_min[inputs].to_numpy()
unscaled_ub = x_data_max[inputs].to_numpy()
#output
output_lb = y_data_min[outputs].to_numpy()
output_ub = y_data_max[outputs].to_numpy()

# saving scaled bounds
# input
scaled_lb = x_scaled_min[inputs].to_numpy()
scaled_ub = x_scaled_max[inputs].to_numpy()
# output
scaled_output_lb = y_scaled_min[outputs].to_numpy()
scaled_output_ub = y_scaled_max[outputs].to_numpy()

scaled_input_bounds = {i: (scaled_lb[i], scaled_ub[i]) for i in range(len(inputs))}
unscaled_input_bounds = {i: (unscaled_lb[i], unscaled_ub[i]) for i in range(len(inputs))}

In [12]:
# Optimisation Data
obj_scaler = 1e-6

# process parameters
feed_conc_val = 0.05 # ppb
EPA_conc_val = 9.66e-9 # mol/m3
capacity_val = 1000 # m3/h

# materials properties
density_val = 501 # kg/m3
porosity_val = 0.4
dp_val = 0.002 # m
qm1_val = 0.5 # mol/kg
kl1_val = 10 # m3/mol
sorbent_cost_val = 2 # $/kg

# loading saved nn
nn_PFAS_model = keras.models.load_model("ALLMAT_relu_mse.keras", compile=False)
net = load_keras_sequential(
        nn_PFAS_model, scaling_object=scaler, scaled_input_bounds=scaled_input_bounds, unscaled_input_bounds=unscaled_input_bounds
    )

# Pyomo model with algebraic constraints
def solve_model(objective='cost', eps_emissions=None, feed_conc=feed_conc_val, EPA_conc=EPA_conc_val, capacity=capacity_val, density=density_val, porosity=porosity_val, dp=dp_val, qm1=qm1_val, kl1=kl1_val, sorbent_cost=sorbent_cost_val):

    max_capacity_per_bed = 120 # m3/h
    N_beds = int(np.ceil(capacity/max_capacity_per_bed))
    Q_per_column = capacity/N_beds

    optimization_model = pyo.ConcreteModel()
    
    optimization_model.diameter = pyo.Var(initialize=(unscaled_lb[1]+unscaled_ub[1])/2, bounds=(unscaled_lb[1], unscaled_ub[1]))
    optimization_model.column_height = pyo.Var(initialize=(unscaled_lb[2]+unscaled_ub[2])/2, bounds=(unscaled_lb[2], unscaled_ub[2]))
    optimization_model.porosity = pyo.Var(initialize=(unscaled_lb[3]+unscaled_ub[3])/2, bounds=(unscaled_lb[3], unscaled_ub[3]))
    optimization_model.dp = pyo.Var(initialize=(unscaled_lb[7]+unscaled_ub[7])/2, bounds=(unscaled_lb[7], unscaled_ub[7]))
    optimization_model.ads_time = pyo.Var(initialize=(unscaled_lb[4]+unscaled_ub[4])/2, bounds=(unscaled_lb[4], unscaled_ub[4]))
    
    optimization_model.total_LCP = pyo.Var(bounds=(output_lb[0], output_ub[0]))
    optimization_model.emissions = pyo.Var(bounds=(output_lb[1], output_ub[1]))
    optimization_model.PFAS_eff_conc = pyo.Var(bounds=(output_lb[2], output_ub[2]))

    optimization_model.nn = OmltBlock()
    formulated_problem = ReluBigMFormulation(net)
    optimization_model.nn.build_formulation(formulated_problem)
    
    optimization_model.nn.inputs[0].fix(pyo.log(capacity))
    optimization_model.nn.inputs[5].fix(pyo.log(feed_conc))
    optimization_model.nn.inputs[6].fix(density)
    optimization_model.nn.inputs[8].fix(pyo.log(qm1))
    optimization_model.nn.inputs[9].fix(pyo.log(kl1))
    optimization_model.nn.inputs[10].fix(pyo.log(sorbent_cost))

    optimization_model.con_in_d = pyo.Constraint(expr= optimization_model.diameter == optimization_model.nn.inputs[1])
    optimization_model.con_in_H = pyo.Constraint(expr= optimization_model.column_height == optimization_model.nn.inputs[2])
    optimization_model.con_in_p = pyo.Constraint(expr= optimization_model.porosity == optimization_model.nn.inputs[3])
    optimization_model.con_in_dp = pyo.Constraint(expr= optimization_model.dp == optimization_model.nn.inputs[7])
    optimization_model.con_in_t = pyo.Constraint(expr= optimization_model.ads_time == optimization_model.nn.inputs[4])

    optimization_model.con_out_LCP = pyo.Constraint(expr= optimization_model.total_LCP == optimization_model.nn.outputs[0])
    optimization_model.con_out_emi = pyo.Constraint(expr= optimization_model.emissions == optimization_model.nn.outputs[1])
    optimization_model.con_out_eff = pyo.Constraint(expr= optimization_model.PFAS_eff_conc == optimization_model.nn.outputs[2])

    # constraints
    optimization_model.epa_limit = pyo.Constraint(expr= optimization_model.PFAS_eff_conc <= pyo.log(EPA_conc + epsilon))
    optimization_model.min_velocity = pyo.Constraint(expr= 4*Q_per_column >= 2*(np.pi*optimization_model.diameter**2))
    optimization_model.max_velocity = pyo.Constraint(expr= 4*Q_per_column <= 15*(np.pi*optimization_model.diameter**2))

    if eps_emissions is not None:
        optimization_model.emissions_con = pyo.Constraint(expr= optimization_model.emissions <= eps_emissions)
    
    if objective == 'cost':
        optimization_model.obj = pyo.Objective(expr=(optimization_model.total_LCP), sense=pyo.minimize)
    else:
        optimization_model.obj = pyo.Objective(expr=(optimization_model.emissions), sense=pyo.minimize)

    options = {
        'MaxTime': 500,
        'epsr': 0.05,
        'epsa': 1e-4,
        'lpsol': 3,
        'cplexlibname': '"C:/Users/huberttreve/Documents/Python/Solvers/CPLEX/cplex/bin/x64_win64/cplex2212.dll"',
        'threads': 16,
        'prlevel': 1,
        'dolocal': 1
        }

    # print("# of variables: ", optimization_model.nvariables())
    # print("# of constraints: ", optimization_model.nconstraints())
    
    solver = pyo.SolverFactory('baron') 
    results = solver.solve(optimization_model, tee=False, options=options)
    
    term_cond = str(results.solver.termination_condition)
    
    # Silently check if the objective has a value
    obj_val = pyo.value(optimization_model.obj, exception=False)
    solution_found = (obj_val is not None)

    if term_cond == 'optimal':
        # print("Success: Global optimal solution found")
        status = "Optimal"
    elif term_cond == 'maxTimeLimit' and solution_found:
        # print(f"Success: Time limit hit. Feasible solution found (Objective: {obj_val:.4f}).")
        status = "Feasible (Time Limit)"
    else:
        # print(f"Solver stopped ('{term_cond}') and NO feasible solution was found.")
        status = "Infeasible"

    def get_val(var):
        # Silently check each variable, return NaN if empty
        v = pyo.value(var, exception=False)
        return v if v is not None else np.nan

    return (capacity,
            N_beds,
            get_val(optimization_model.diameter),
            get_val(optimization_model.column_height),
            get_val(optimization_model.porosity),
            np.exp(get_val(optimization_model.dp)),
            np.exp(get_val(optimization_model.ads_time)),
            feed_conc,
            density,
            qm1,
            kl1,
            sorbent_cost,
            np.exp(get_val(optimization_model.total_LCP)),
            np.exp(get_val(optimization_model.emissions)),
            np.exp(get_val(optimization_model.PFAS_eff_conc) - epsilon),
            status)

In [13]:
import itertools
import time

def run_thermodynamic_sweep(capacity_val, sorbent_cost_val, qm1_bounds, kl1_bounds, grid_size=15):
    """
    Runs a 2D parametric sweep of q_max vs K_L for a fixed capacity and sorbent cost.
    Saves the output to a dynamically named Excel file.
    """
    qm1_min, qm1_max = qm1_bounds
    kl1_min, kl1_max = kl1_bounds
    
    # Generate linearly spaced values for the sweep
    qm1_values = np.linspace(qm1_min, qm1_max, grid_size)
    kl1_values = np.linspace(kl1_min, kl1_max, grid_size)
    
    run_grid = list(itertools.product(qm1_values, kl1_values))
    total_runs = len(run_grid)
    
    print(f"\n========================================================")
    print(f"STARTING SWEEP: Capacity = {capacity_val} m3/h | Cost = ${sorbent_cost_val}/kg")
    print(f"Grid: {grid_size}x{grid_size} ({total_runs} total optimization runs)")
    print(f"========================================================\n")
    
    results_data = []
    start_time = time.time()

    for idx, (q_val, kl_val) in enumerate(run_grid):
        print(f"Run {idx+1}/{total_runs} | q_max: {q_val:.2f} | K_L: {kl_val:.2f}")
        
        try:
            # Pass BOTH qm1 and kl1 to the solver, along with the fixed parameters
            res = solve_model(
                objective='cost',
                eps_emissions=None,
                capacity=capacity_val,
                sorbent_cost=sorbent_cost_val,
                qm1=q_val,
                kl1=kl_val
            )
            results_data.append(res)
            
        except Exception as e:
            print(f"  -> ⚠️ Solver crashed (Out-of-bounds). Error: {e}")
            
            # Ensure the failed tuple strictly matches your column order
            failed_res = (
                capacity_val,    # capacity
                np.nan,          # N_beds
                np.nan,          # diameter
                np.nan,          # column_height
                np.nan,          # porosity
                np.nan,          # dp
                np.nan,          # ads_time
                np.nan,          # feed_conc
                np.nan,          # density
                q_val,           # qm1 (Swept)
                kl_val,          # kl1 (Swept)
                sorbent_cost_val,# sorbent_cost (Fixed)
                np.nan,          # total_LCP
                np.nan,          # emissions
                np.nan,          # PFAS_eff_conc
                "Solver Crashed" # status
            )
            results_data.append(failed_res)

    # Convert to DataFrame
    columns = [
        "capacity", "N_beds", "diameter", "column_height", "porosity", "dp", 
        "ads_time", "feed_conc", "density", "qm1", "kl1", "sorbent_cost",
        "total_LCP", "emissions", "PFAS_eff_conc", "status"
    ]
    df_results = pd.DataFrame(results_data, columns=columns)
    
    # Generate a unique filename based on the fixed parameters
    excel_filename = f"optimisation_qmax_vs_kl_Cap{capacity_val}_Cost{sorbent_cost_val}.xlsx"
    df_results.to_excel(excel_filename, index=False)
    
    elapsed_mins = (time.time() - start_time) / 60
    print(f"\n✅ Sweep complete in {elapsed_mins:.1f} mins. Saved to {excel_filename}")
    return df_results

# ==========================================
# MASTER EXECUTION LOOP
# ==========================================
if __name__ == "__main__":
    
    # 1. Define your thermodynamic bounds (extract these from your LHS df1 if preferred)
    qm1_min = df1['qm1'].min()  # mol/kg
    qm1_max = df1['qm1'].max()  # mol/kg
    KL1_min = df1['kl1'].min()  # m3/mol
    KL1_max = df1['kl1'].max()  # m3/mol
    QM1_BOUNDS = (qm1_min, qm1_max)    # (min, max) mol/kg
    KL1_BOUNDS = (KL1_min, KL1_max)    # (min, max) m3/mol
    
    # 2. Define the fixed scenarios you want to test
    capacities_to_test = [1000]
    sorbent_costs_to_test = [2, 10, 100, 1000] 
    
    # 3. Run the batch sweeps
    # Note: A 15x15 grid is 225 runs per combination. 
    # 2 capacities * 3 costs = 6 separate files (1,350 total Pyomo/BARON solves)
    for cap in capacities_to_test:
        for cost in sorbent_costs_to_test:
            run_thermodynamic_sweep(
                capacity_val=cap, 
                sorbent_cost_val=cost, 
                qm1_bounds=QM1_BOUNDS, 
                kl1_bounds=KL1_BOUNDS, 
                grid_size=20  # Adjust this depending on how fast BARON solves
            )


STARTING SWEEP: Capacity = 1000 m3/h | Cost = $2/kg
Grid: 20x20 (400 total optimization runs)

Run 1/400 | q_max: 0.00 | K_L: 1.00
Run 2/400 | q_max: 0.00 | K_L: 527.26
Run 3/400 | q_max: 0.00 | K_L: 1053.52
Run 4/400 | q_max: 0.00 | K_L: 1579.78
Run 5/400 | q_max: 0.00 | K_L: 2106.04
Run 6/400 | q_max: 0.00 | K_L: 2632.30
Run 7/400 | q_max: 0.00 | K_L: 3158.56
Run 8/400 | q_max: 0.00 | K_L: 3684.82
Run 9/400 | q_max: 0.00 | K_L: 4211.08
Run 10/400 | q_max: 0.00 | K_L: 4737.34
Run 11/400 | q_max: 0.00 | K_L: 5263.60
Run 12/400 | q_max: 0.00 | K_L: 5789.86
Run 13/400 | q_max: 0.00 | K_L: 6316.12
Run 14/400 | q_max: 0.00 | K_L: 6842.38
Run 15/400 | q_max: 0.00 | K_L: 7368.64
Run 16/400 | q_max: 0.00 | K_L: 7894.90
Run 17/400 | q_max: 0.00 | K_L: 8421.16
Run 18/400 | q_max: 0.00 | K_L: 8947.42
Run 19/400 | q_max: 0.00 | K_L: 9473.68
Run 20/400 | q_max: 0.00 | K_L: 9999.94
Run 21/400 | q_max: 0.53 | K_L: 1.00
Run 22/400 | q_max: 0.53 | K_L: 527.26
Run 23/400 | q_max: 0.53 | K_L: 1053.52
R